# End-to-End MLOps: Heart Disease Classifier

This notebook is a guided learning version of the complete assignment. It explains **why** each step exists and demonstrates the same reusable modules used by the production scripts.

## Learning path

1. Define the problem and inspect the dataset.
2. Clean the target and perform EDA.
3. Build leakage-safe preprocessing and compare models.
4. Understand MLflow experiment tracking and model packaging.
5. Run local inference and call the FastAPI service.
6. Connect the API to Docker, Kubernetes, and monitoring.

The notebook is for understanding. The script files remain the reproducible source of truth used by CI/CD.

## 1. Problem definition

The UCI Heart Disease dataset contains patient measurements and a target representing disease severity. The source target has values from 0 to 4, while this assignment asks for binary classification.

We use this mapping:

- `0`: no disease, binary class `0`
- `1`, `2`, `3`, `4`: disease present, binary class `1`

This is an educational experiment, not a clinical diagnostic system.

In [ ]:
import json
import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from app.config import ARTIFACTS_DIR, EDA_OUTPUT_DIR, FEATURE_COLUMNS, RAW_DATA_PATH, TARGET_COLUMN
from app.data import load_raw_data, prepare_training_data, validate_schema

EDA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
raw = load_raw_data(RAW_DATA_PATH)
validate_schema(raw)
print(f"Rows: {len(raw)}")
print(f"Features: {len(FEATURE_COLUMNS)}")
raw.head()

## 2. Data quality and target preparation

We inspect missing values before modeling. Missing values must be handled consistently, so the final model pipeline performs imputation instead of dropping records. Keeping the raw dataset unchanged also preserves provenance.

In [ ]:
print("Missing values by column:")
display(raw.isna().sum().sort_values(ascending=False))

binary_target = (raw[TARGET_COLUMN] > 0).astype(int)
print("Binary class counts:")
display(binary_target.value_counts().sort_index())

## 3. Exploratory data analysis

EDA is used to understand the data before selecting preprocessing and models:

- Histograms show feature scale and skew.
- A class-balance chart shows whether stratification is needed.
- A correlation heatmap shows descriptive relationships, not causation.

The production equivalent is `python -m scripts.eda`.

In [ ]:
class_counts = binary_target.value_counts().sort_index()
class_counts.plot.bar(title="Class balance", color=["#4c78a8", "#f58518"])
plt.xlabel("Class: 0=no disease, 1=disease")
plt.ylabel("Rows")
plt.tight_layout()
plt.savefig(EDA_OUTPUT_DIR / "class_balance_notebook.png", dpi=150)
plt.show()

raw[FEATURE_COLUMNS].hist(figsize=(14, 10), bins=15)
plt.suptitle("Feature distributions")
plt.tight_layout()
plt.savefig(EDA_OUTPUT_DIR / "feature_distributions_notebook.png", dpi=150)
plt.show()

analysis = raw[FEATURE_COLUMNS].copy()
analysis[TARGET_COLUMN] = binary_target
plt.figure(figsize=(12, 9))
sns.heatmap(analysis.corr(numeric_only=True), cmap="vlag", center=0)
plt.title("Feature and target correlations")
plt.tight_layout()
plt.savefig(EDA_OUTPUT_DIR / "correlation_heatmap_notebook.png", dpi=150)
plt.show()

## 4. Feature engineering and leakage prevention

The project uses a scikit-learn pipeline:

- Numeric values: median imputation followed by standard scaling.
- Categorical values: most-frequent imputation followed by one-hot encoding.

Putting preprocessing and the classifier in one pipeline is important. It prevents test-fold information from leaking into training and guarantees that the API applies the same transformations as training.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from app.modeling import RANDOM_STATE, build_pipeline, candidate_models

X, y = prepare_training_data(raw)
X_train, X_test, y_train, y_test = train_test_split(
+        X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print("Training rows:", len(X_train))
print("Holdout rows:", len(X_test))

## 5. Model comparison

We compare two complementary classifiers:

- Logistic Regression: an interpretable linear baseline.
- Random Forest: a nonlinear ensemble that can learn interactions.

Five-fold stratified cross-validation gives a more stable estimate than relying on one split. ROC-AUC is the primary selection metric, while accuracy, precision, and recall are also reported.

In [ ]:
scoring = {"accuracy": "accuracy", "precision": "precision", "recall": "recall", "roc_auc": "roc_auc"}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rows = []
for name, classifier in candidate_models().items():
    pipeline = build_pipeline(classifier)
    scores = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=scoring)
    rows.append({"model": name, **{metric: scores[f"test_{metric}"].mean() for metric in scoring}})

comparison = pd.DataFrame(rows).sort_values("roc_auc", ascending=False)
display(comparison)
champion_name = comparison.iloc[0]["model"]
champion = build_pipeline(candidate_models()[champion_name])
champion.fit(X_train, y_train)
print("Selected champion:", champion_name)

### Metric interpretation

- **Accuracy:** total correct predictions.
- **Precision:** reliability of positive predictions.
- **Recall:** proportion of actual disease cases found.
- **ROC-AUC:** ranking quality over possible thresholds.

Recall is important in a risk-screening example because missed positive cases can be costly. These metrics are evidence for this experiment only, not clinical validation.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

holdout_prediction = champion.predict(X_test)
holdout_probability = champion.predict_proba(X_test)[:, 1]
holdout_metrics = dict(
    accuracy=accuracy_score(y_test, holdout_prediction),
    precision=precision_score(y_test, holdout_prediction),
    recall=recall_score(y_test, holdout_prediction),
    roc_auc=roc_auc_score(y_test, holdout_probability),
)
holdout_metrics

## 6. MLflow and reproducibility

The production training script creates one MLflow run per candidate. Each run records parameters, cross-validation metrics, holdout metrics, plots, model artifacts, and a dataset hash.

Start the MLflow UI from the project root:

```bash
mlflow ui --backend-store-uri file:./mlruns
```

Open `http://127.0.0.1:5000` and compare the Logistic Regression and Random Forest runs. The saved `model.joblib` contains preprocessing plus the classifier, which avoids training-serving skew.

In [ ]:
model_path = ARTIFACTS_DIR / "model.joblib"
model = joblib.load(model_path)
print("Loaded production artifact:", model_path)
print("Artifact prediction matches notebook pipeline:", (model.predict(X_test) == champion.predict(X_test)).all())

## 7. Local inference

The API receives raw JSON, converts it to a one-row DataFrame, and passes it to the saved pipeline. The confidence is the probability of the predicted class.

In [ ]:
patient = dict(
    age=63, sex=1, cp=1, trestbps=145, chol=233,
    fbs=1, restecg=2, thalach=150, exang=0, oldpeak=2.3,
    slope=3, ca=0, thal=6,
)
request_frame = pd.DataFrame([patient], columns=FEATURE_COLUMNS)
prediction = int(model.predict(request_frame)[0])
probabilities = model.predict_proba(request_frame)[0]
confidence = float(probabilities[prediction])
print({"prediction": prediction, "confidence": round(confidence, 6)})

## 8. FastAPI, Docker, Kubernetes, and monitoring

Start the local API from the project root:

```bash
uvicorn app.api:app --host 127.0.0.1 --port 8000
```

Endpoints:

- `GET /health`: readiness check.
- `POST /predict`: prediction and confidence.
- `GET /metrics`: Prometheus-compatible request counters and latency histograms.

The Dockerfile packages the API and model into a reproducible image. `deployment/k8s.yaml` adds a Kubernetes Deployment, Service, readiness probe, liveness probe, and resource limits.

In [ ]:
import requests

try:
    health = requests.get("http://127.0.0.1:8000/health", timeout=2)
    metrics = requests.get("http://127.0.0.1:8000/metrics", timeout=2)
    print("Health:", health.status_code, health.json())
    print("Metrics status:", metrics.status_code)
    print(metrics.text[:500])
except requests.RequestException:
    print("The API is not running. Start uvicorn using the command above, then rerun this cell.")

## Final execution checklist

The production sequence is:

```bash
python -m scripts.download_data
python -m scripts.prepare_data
python -m scripts.eda
python -m scripts.train --tracking-uri file:./mlruns
python -m pytest -q
ruff check app scripts tests
docker build -t heart-disease-api:main .
kubectl apply -f deployment/k8s.yaml
```

This notebook helps explain each stage; the scripts, tests, Dockerfile, CI workflow, and Kubernetes manifest provide the production-style implementation.